# 02b - Merge OTP rows to line, trip, and schedule

Reads the cleaned OTP data (`01_otp_load.ipynb`) and the GTFS checkpoints
(`02_gtfs_full_fetch.ipynb`), resolves each OTP row's `line`/`trip_id`/`direction_id`/scheduled-time
columns, and merges them on. Output is saved to `data/` as `2_df_gtfs_linked.parquet`, which everything downstream builds on.

**Note**: `block_id` contains info about plenty of individual train runs, but also actually represents single trains running trips on multiple lines, one immediately after the other (through-running via Center City tunnel). E.g. a single train might complete an Airport run -> Suburban and immediately begin a run from Suburban outbound on the Warminster Line. 59.9% of train_numbers serve more than one line per day.

A naive `merge_asof(..., by="train_number")` can only return one `line` per `(train_number, service_date)` -- every OTP ping for such a train on a given day would get assigned to a single line, sweeping up observations that actually belong to the other line. 

**Fix**: resolve `line`/`trip_id`/`direction_id` per OTP row instead of per `(train_number, day)`, using actual schedule
information from GTFS. For trains with more than one candidate line on a given day, compare each
observation's schedule-implied time (`time - lateness`, SEPTA-3am-service-day-adjusted) against each
candidate's scheduled `[origin, terminus]` window (from `stop_times`), and assign whichever window
actually contains it -- falling back to the nearest window edge for the small number of rows outside all
candidate windows. Validated at full 9-year scale: 99.3% overall match rate.


In [1]:
import pandas as pd
import numpy as np
import gc

BASEPATH = "../data"

## Load reference data

Small crosswalk tables: loaded and reused across every year's OTP chunk below.

In [2]:
# raw OTP data, cleaned in 01_otp_load.ipynb
otp_path = f"{BASEPATH}/1_df_clean.parquet"

# crosswalk: train_number/line/service_id/trip_id/direction_id per release
crosswalk = pd.read_parquet(f"{BASEPATH}/2_gtfs_linkages_since_2017_clean.parquet")
crosswalk["line"] = crosswalk["line"].str.replace(" Line$", "", regex = True)

# which service_id(s) were active on each calendar date, per release
active_service_dates = pd.read_parquet(f"{BASEPATH}/2_gtfs_active_service_dates.parquet")

# raw per-release stop_times, for computing each trip's scheduled window
stop_times = pd.read_parquet(f"{BASEPATH}/2_gtfs_stop_times.parquet")
stop_times["stop_sequence"] = stop_times["stop_sequence"].astype(int)

print(f"crosswalk: {len(crosswalk):,} rows")
print(f"active_service_dates: {len(active_service_dates):,} rows")
print(f"stop_times: {len(stop_times):,} rows")

crosswalk: 366,522 rows
active_service_dates: 34,009 rows
stop_times: 5,392,855 rows


## Derive per-trip scheduled features

One row per `(trip_id, gtfs_date)` for use in sports and peak window calculations later.

In [3]:
def gtfs_time_to_sec(t):
    """
    HH:MM:SS -> seconds since midnight. 
    Not datetime bc of SEPTA's 3a service day"""
    h, m, s = t.split(":")
    return int(h) * 3600 + int(m) * 60 + int(s)

stop_times = stop_times.sort_values(["trip_id", 
                                     "gtfs_date", 
                                     "stop_sequence"])

trip_features = (
    stop_times
    .groupby(["trip_id", "gtfs_date"])
    .agg(
        origin_stop_id = ("stop_id", "first"),
        terminus_stop_id = ("stop_id", "last"),
        sched_origin_sec = ("departure_time", "first"),
        sched_terminus_sec = ("arrival_time", "last"),
        n_scheduled_stops = ("stop_sequence", "count"),
    )
    .reset_index()
    .rename(columns = {"gtfs_date": "source_gtfs_date"})
)

trip_features["sched_origin_sec"] = trip_features["sched_origin_sec"].apply(gtfs_time_to_sec)
trip_features["sched_terminus_sec"] = trip_features["sched_terminus_sec"].apply(gtfs_time_to_sec)
trip_features["sched_duration_sec"] = trip_features["sched_terminus_sec"] - trip_features["sched_origin_sec"]

print(f"{len(trip_features):,} (trip_id, source_gtfs_date) combos")
del stop_times
gc.collect()

363,876 (trip_id, source_gtfs_date) combos


0

## Resolve which release/service_id applies to each service_date

`gtfs_date` doesn't always line up with actual calendar/schedule validity, so this falls back to the
nearest earlier release when a release's own calendar hasn't taken effect yet.

In [4]:
# one row per (service_date, source_gtfs_date) combo
release_calendar_coverage = (
    active_service_dates[["service_date", "source_gtfs_date"]]
    .drop_duplicates()
)
# only releases published at or before the service_date are eligible --
# new releases often just assert the same patterns,
# so among the eligible ones we want the most recent.
release_calendar_coverage = release_calendar_coverage[
    release_calendar_coverage["source_gtfs_date"] <= release_calendar_coverage["service_date"]
]
resolved_release = (
    release_calendar_coverage
    .sort_values("source_gtfs_date")
    .groupby("service_date", as_index = False)
    .last()
)
print(f"Resolved release for {len(resolved_release):,} / "
      f"{active_service_dates['service_date'].nunique():,} distinct service_dates")

# active service_id(s) under that resolved release, for each service_date
active_svc_for_release = (
    active_service_dates
    .merge(resolved_release, on = ["service_date", "source_gtfs_date"], how = "inner")
    [["service_date", "source_gtfs_date", "service_id"]]
    .drop_duplicates()
)
print(f"{len(active_svc_for_release):,} (service_date, service_id) combos after resolution "
      f"({active_svc_for_release.groupby('service_date').size().mean():.2f} service_ids/date on average)")

Resolved release for 3,832 / 3,837 distinct service_dates
5,252 (service_date, service_id) combos after resolution (1.37 service_ids/date on average)


## Resolve line/trip_id/direction_id per OTP row

For each `(service_date, train_number)`, gather every crosswalk candidate matching the resolved release +
active service_id(s) -- one candidate for a normal train, two or more for a multi-run train. Then compute schedule-implied time for each OTP observation and assign whichever candidate's scheduled window contains
it. Fall back to the nearest window edge if none overlap directly. Chunked by year for RAM mgmt.

In [5]:
years = sorted(
    pd.read_parquet(otp_path, columns = ["service_date"])["service_date"].dt.year.unique()
)

resolved_all = []

for year in years:
    chunk = pd.read_parquet(
        otp_path,
        columns = ["service_date", "train_number", "time", "lateness", "datetime"],
        filters = [("service_date", ">=", pd.Timestamp(f"{year}-01-01")),
                   ("service_date", "<", pd.Timestamp(f"{year + 1}-01-01"))],
    )
    chunk["train_number"] = chunk["train_number"].astype(str)
    chunk["_row_id"] = np.arange(len(chunk))

    # distinct (service_date, train_number) combos this year -- resolve
    # once per combo
    combos = chunk[["service_date", "train_number"]].drop_duplicates()
    combos = combos.merge(active_svc_for_release, on = "service_date", how = "left")

    candidates = combos.merge(
        crosswalk[["train_number", "service_id", "gtfs_date",
                   "trip_id", "line", "direction_id"]]
            .rename(columns = {"gtfs_date": "source_gtfs_date"}),
        on = ["train_number", "service_id", "source_gtfs_date"],
        how = "inner",
    )
    candidates = candidates.merge(
        trip_features, on = ["trip_id", "source_gtfs_date"], how = "left"
    )

    # fan the entire row-level group against every candidate for its
    # (service_date, train_number)
    # should be 1:1 for normal trains, 1:N for multi-run
    fanned = chunk.merge(
        candidates, on = ["service_date", "train_number"], how = "left"
    )

    # schedule-implied time: undo the observed delay to get back to what
    # the schedule would say, using the SEPTA 3a-adjusted value
    # so overnight runs aren't misclassified
    fanned["_time_adj"] = (
        fanned["datetime"]
        - fanned["service_date"].dt.tz_localize(fanned["datetime"].dt.tz)
    ).dt.total_seconds()
    fanned["_sched_implied_sec"] = fanned["_time_adj"] - fanned["lateness"] * 60

    fanned["_contains"] = (
        (fanned["sched_origin_sec"] <= fanned["_sched_implied_sec"])
        & (fanned["_sched_implied_sec"] <= fanned["sched_terminus_sec"])
    )
    fanned["_dist"] = np.where(
        fanned["_contains"], 0,
        np.maximum(
            fanned["sched_origin_sec"] - fanned["_sched_implied_sec"],
            np.maximum(fanned["_sched_implied_sec"] - fanned["sched_terminus_sec"], 0)
        )
    )
    # rows with no candidate (crosswalk match failed)
    # have NA sched_origin_sec/_dist
    # sort NaN last so a real candidate always wins over "no match"
    # and only rows with actually 0 candidates end up unmatched
    fanned = fanned.sort_values(["_row_id", "_dist"], na_position = "last")
    resolved = fanned.drop_duplicates(subset = "_row_id", keep = "first")

    resolved = resolved[[
        "service_date", "train_number", "time", "line", "trip_id",
        "direction_id", "source_gtfs_date", "sched_origin_sec",
        "sched_terminus_sec", "sched_duration_sec", "n_scheduled_stops",
        "origin_stop_id", "terminus_stop_id",
    ]]
    resolved_all.append(resolved)

    matched = resolved["line"].notna()
    multi_candidate = (
        candidates.groupby(["service_date", "train_number"])["trip_id"].nunique() > 1
    ).mean()
    print(f"{year}: {len(resolved):,} rows, {matched.mean()*100:.1f}% matched, "
          f"{multi_candidate*100:.1f}% of train-days had >1 candidate line")

    del chunk, combos, candidates, fanned, resolved
    gc.collect()

resolved_lines = pd.concat(resolved_all, ignore_index = True)
del resolved_all
gc.collect()

2017: 2,315,468 rows, 99.3% matched, 61.9% of train-days had >1 candidate line


2018: 1,480,001 rows, 98.6% matched, 61.2% of train-days had >1 candidate line


2019: 1,990,712 rows, 99.4% matched, 60.5% of train-days had >1 candidate line


2020: 1,121,532 rows, 98.9% matched, 56.9% of train-days had >1 candidate line


2021: 1,580,251 rows, 99.2% matched, 57.5% of train-days had >1 candidate line


2022: 1,859,407 rows, 98.8% matched, 55.4% of train-days had >1 candidate line


2023: 2,871,273 rows, 99.5% matched, 54.7% of train-days had >1 candidate line


2024: 3,428,108 rows, 99.6% matched, 60.5% of train-days had >1 candidate line


2025: 3,484,652 rows, 99.3% matched, 56.4% of train-days had >1 candidate line


0

## Validate

Spot-check a known through-running train (420: Airport -> Warminster) on a real date, and confirm we've actually split its pings between the two lines instead of collapsing to one.

In [6]:
otp_full = pd.read_parquet(otp_path)
otp_full["train_number"] = otp_full["train_number"].astype(str)

check = otp_full.merge(
    resolved_lines, on = ["service_date", "train_number", "time"], how = "left"
)

print("Overall match rate:", f"{check['line'].notna().mean()*100:.1f}%")
print()

sample = check[
    (check["train_number"] == "420") & (check["service_date"] == "2025-06-02")
].sort_values("time")
print("train 420, 2025-06-02 -- line assignment per ping:")
print(sample[["time", "lateness", "line", "trip_id", "direction_id"]].to_string(index = False))
print()
print("Distinct lines this train shows on this day (should be 2):",
      sample["line"].nunique())

Overall match rate: 99.3%



train 420, 2025-06-02 -- line assignment per ping:
    time  lateness       line       trip_id direction_id
09:38:01         0    Airport AIR_420_C50_M            0
09:43:01         1    Airport AIR_420_C50_M            0
09:44:01         2    Airport AIR_420_C50_M            0
09:47:01         1    Airport AIR_420_C50_M            0
09:53:01         2    Airport AIR_420_C50_M            0
09:56:01         3    Airport AIR_420_C50_M            0
09:57:01         4    Airport AIR_420_C50_M            0
09:59:01         3    Airport AIR_420_C50_M            0
10:00:02         2    Airport AIR_420_C50_M            0
10:03:01         1    Airport AIR_420_C50_M            0
10:04:01         2    Airport AIR_420_C50_M            0
10:05:01         3    Airport AIR_420_C50_M            0
10:07:01         1 Warminster WAR_420_C50_M            0
10:10:02         2 Warminster WAR_420_C50_M            0
10:11:47         0 Warminster WAR_420_C50_M            0
10:21:01         1 Warminster WAR_420

## Save

Save merged dataframe to `data/` as `2_df_gtfs_linked.parquet`. Downstream notebooks read from here.

In [7]:
check.to_parquet(f"{BASEPATH}/2_df_gtfs_linked.parquet")
print(f"Saved {len(check):,} rows")

Saved 20,131,404 rows
